# Visible-features bluff model (what you could write down at the table)

This is **not** the same model as the street win-% RF in notebook 02. Here the inputs are **only** things a spectator sees:
- community cards, pot, stacks in BB, SPR-ish geometry
- how much **this seat** has put in (`bet_*` totals in the CSV; live ops in the UI)
- street phase + rough aggression (`raise` mentions in action text for history rows)

**No hole cards** go into the vector — we always call `build_stage_feature_payload` with **empty** hole cards so strengths/draws come from the board + betting context only.

**Label:** `is_bluffing` from the cleaning pipeline (`label_bluffs` — low deterministic strength + aggressive actions). It’s a weak proxy, but it trains something directional.

**Train:** from repo root run `python visible_bluff_train.py` after `data/cleanedGambling.csv` exists. That writes `visible_bluff_model.pkl` + `visible_bluff_feature_names.pkl`. The poker UI reads those and paints **“Bluff ~xx%”** above each seat.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier

from scripts.features.visible_bluff_features import VISIBLE_BLUFF_FEATURES, vector_from_csv_row

sns.set_theme(style="whitegrid")

df = pd.read_csv(ROOT / "data" / "cleanedGambling.csv")
sample = df.sample(min(12000, len(df)), random_state=42)
rows = [vector_from_csv_row(r) for _, r in sample.iterrows()]
train_y = sample["is_bluffing"].astype(int).tolist()
X = pd.DataFrame(rows).reindex(columns=VISIBLE_BLUFF_FEATURES).fillna(0.0)
y = pd.Series(train_y)
rf = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=14, min_samples_leaf=8)
rf.fit(X, y)
imp = pd.DataFrame({"feature": VISIBLE_BLUFF_FEATURES, "importance": rf.feature_importances_}).sort_values("importance")
fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=imp, y="feature", x="importance", ax=ax, color="#9B59B6")
ax.set_title("Notebook RF importance — observable bluff features (diagnostic)")
plt.tight_layout()
plt.show()
print("Train the real calibrated model with: python visible_bluff_train.py")
print(imp.tail(6).to_string(index=False))

### Where it shows up
`scripts/ui/poker_table_tab.py` calls `predict_visible_bluff_probability` per seat. If the pickle files are missing the UI silently skips the labels — train first.